# Tome 12, Chapter 8: Zeta functions and densities (Python companion)

Exercises first, solutions at the end. We count ideals and watch the class number formula appear, compute relative class numbers of cyclotomic fields from Bernoulli numbers and hunt irregular primes as Kummer did, test Chebotarev's theorem on several Galois groups, and end with its use in computer algebra: recognising a Galois group, and proving irreducibility, by sampling Frobenius elements. Requires `cypari2`.

In [ ]:
import math, random, cmath
from collections import Counter
from fractions import Fraction
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2 * 10**9, silent=True)
random.seed(8)

## Exercises

**P8.1 (★) The class number formula, numerically.** For a few fields $K$ (quadratic, cubic, quartic), compute $h_K$, $R_K$, $w_K$, $d_K$ with `bnfinit`, the constant $h_K\kappa_K$ of Theorems 8.5 and 8.9, and compare with the number of ideals of norm at most $x$ (coefficients of $\zeta_K$ from `dirzetak`), divided by $x$, for $x=10^5$. How fast does the ratio converge? Compare the error with $x^{1-1/n}$.

**P8.2 (★★) Kummer's table.** Compute $h^-=2p\prod_{\chi\text{ odd}}\bigl|\frac12B_{1,\chi}\bigr|$ (Theorem 8.13) exactly for $p<70$, using the Teichmüller-type character $\chi(g^j)=\zeta_{p-1}^j$ and rational arithmetic in $\mathbb Q(\zeta_{p-1})$ (or a high-precision complex product rounded to an integer). Check against `bnfinit(polcyclo(p))` for $p\le19$ (where $h^+=1$, so $h=h^-$) and against known values. Then list the irregular primes below $1000$ with Kummer's criterion (Theorem 8.16), and compare the proportion of regular primes with $e^{-1/2}$.

**P8.3 (★★) Chebotarev on four groups.** For $X^4-2$ ($D_4$), $X^4+X+1$ ($S_4$), $X^4+8X+12$ ($A_4$), $X^4-5X^2+5$ ($C_4$), compute the factorisation patterns modulo the primes below $10^5$ and compare with the cycle-type proportions of the Galois group (use `polgalois` to name it).

**P8.4 (★★★) CS application: Galois groups and irreducibility by sampling Frobenius.** Computer algebra systems use Chebotarev's theorem as a statistical oracle.
1. *Irreducibility certificates.* If $f\in\mathbb Z[X]$ is reducible, every factorisation pattern modulo $p$ refines the degrees of a fixed factorisation over $\mathbb Q$. Show that for a quartic, seeing the patterns $(1,3)$ and $(2,2)$ (or a $(4)$) proves irreducibility. Write a function that searches primes until such a certificate is found, and measure how many primes it needs on random irreducible quartics. Why does Chebotarev guarantee it terminates for $S_4$?
2. *Galois group recognition.* For each transitive group of degree $4$ ($C_4$, $V_4$, $D_4$, $A_4$, $S_4$), tabulate the probabilities of the patterns $(1,1,1,1)$, $(1,1,2)$, $(2,2)$, $(1,3)$, $(4)$. Given $N$ sampled patterns, choose the group of maximum likelihood (patterns impossible for a group exclude it). How large must $N$ be to identify the group of random quartics correctly $99\%$ of the time? Compare with `polgalois`. Which pair of groups is hardest to separate, and why can sampling never distinguish some groups in higher degree?

**P8.5 (★) The error term.** For $\mathbb Z[\sqrt{-5}]$, plot $A_C(x)-\kappa x$ for both classes up to $x=10^5$, divided by $\sqrt x$. Is $O(x^{1/2})$ the truth?

## Solutions

### P8.1
**How to find it.** `bnfinit` gives `bnf.no` $=h$, `bnf.reg` $=R$, `bnf.tu[1]` $=w$, `nf.disc`, `nf.sign`; `dirzetak(nf, N)` returns the numbers $a_k$ of ideals of norm $k$ for $k\le N$.

In [ ]:
def kappa_h(pol):
    bnf = pari.bnfinit(pari(pol), 1)
    nf = bnf.bnf_get_nf() if hasattr(bnf, 'bnf_get_nf') else bnf
    r1, r2 = [int(x) for x in pari('(b)->b.sign')(bnf)]
    h = int(pari('(b)->b.no')(bnf)); R = float(pari('(b)->b.reg')(bnf))
    w = int(pari('(b)->b.tu[1]')(bnf)); d = int(pari('(b)->b.disc')(bnf))
    return h, R, w, d, r1, r2, h * 2**r1 * (2*math.pi)**r2 * R / (w * math.sqrt(abs(d)))

X = 10**5
for pol in ["x^2+5", "x^2-10", "x^2+23", "x^3-2", "x^3-x-1", "x^4-x-1", "x^4+1"]:
    h, R, w, d, r1, r2, hk = kappa_h(pol)
    a = pari.dirzetak(pari.nfinit(pari(pol)), X)
    A = sum(int(t) for t in a)
    n = r1 + 2 * r2
    print(f"{pol:10s} h={h} R={R:8.4f} w={w} d={d:6d} | h*kappa={hk:.5f}  A(x)/x={A/X:.5f}"
          f"  error/x^(1-1/n)={(A - hk*X)/X**(1-1/n):+.3f}")

The ratios agree with $h\kappa$ to three or four digits, and the normalised error stays of moderate size: the $x^{1-1/n}$ of the Lipschitz principle is a sound (if not sharp) bound. For $\mathbb Q(\sqrt{-23})$, $h\kappa=\frac{3\pi}{\sqrt{23}}=1.9653$, the opening of the chapter in numbers.

### P8.2
**How to find it.** With a primitive root $g$ modulo $p$ and $\chi(g^j)=e^{2\pi ijk/(p-1)}$ for odd $k$, $B_{1,\chi}=\frac1p\sum_j\chi(g^j)(g^j\bmod p)$. The product of $|B_{1,\chi}/2|$ over the odd characters, times $2p$, is an integer; floating point with $50$ digits is plenty for $p<70$ (we use `mpmath`-free PARI complex arithmetic).

In [ ]:
def hminus(p):
    pari.set_real_precision(60)
    g = int(pari.znprimroot(p)); n = p - 1
    pw = [pow(g, j, p) for j in range(n)]
    prod = pari(1)
    for k in range(1, n, 2):
        z = pari(f"exp(2*Pi*I*{k}/{n})")
        B = sum(z**j * pw[j] for j in range(n)) / p
        prod *= pari.abs(B) / 2
    v = 2 * p * prod
    return int(pari.round(v)), float(pari.abs(v - pari.round(v)))

for p in [int(q) for q in pari.primes(20) if 3 <= int(q) < 70]:
    hm, err = hminus(p)
    print(p, hm, f"(rounding error {err:.1e})")
for p in [5, 7, 11, 13, 17, 19]:
    print("bnfinit check p =", p, "h =", pari(f"bnfinit(polcyclo({p}),1).no"))

$h^-=1$ up to $19$, then $3, 8, 9, 37, 121, 211, 695, 4889, 41241, 76301, 853513$ for $p=23,\dots,67$: Kummer computed such values by hand for $p<100$. Now irregular primes: by Kummer's criterion (Theorem 8.16) $p$ divides $h^-$ iff it divides some $B_k$, $k\le p-3$ even.

In [ ]:
def irregular_indices(p):
    return [k for k in range(2, p - 2, 2) if int(pari.numerator(pari.bernfrac(k))) % p == 0]

irr = {}
ps = [int(q) for q in pari.primes(pari.primepi(1000)) if int(q) >= 5]
for p in ps:
    ks = irregular_indices(p)
    if ks: irr[p] = ks
print("irregular primes < 1000:", len(irr))
print({p: irr[p] for p in list(irr)[:12]})
print("Kummer's 1850 range (< 165):", [p for p in irr if p < 165])
print("proportion regular:", round(1 - len(irr) / (len(ps) + 1), 4), " vs e^(-1/2) =", round(math.exp(-0.5), 4))
for p in [37, 59, 67]:
    hm, _ = hminus(p) if p < 70 else (None, None)
    print(p, "h^- =", hm, " divisible by p:", hm % p == 0)

The first irregular primes are $37, 59, 67, 101, 103, 131, 149, 157$: Kummer's table. The heuristic of the think box: each of the $\frac{p-3}2$ numbers $B_k$ is divisible by $p$ with probability $\frac1p$, so $p$ is regular with probability $(1-\frac1p)^{(p-3)/2}\approx e^{-1/2}\approx0.607$. The count below $1000$ is close; Buhler, Crandall and others have checked it up to $2^{31}$ and beyond. And $37\mid h^-(37)=37$, $59\mid h^-(59)=41241=3\cdot59\cdot233$, $67\mid h^-(67)$: the criterion at work.

### P8.3
**How to find it.** `polgalois(f)` names the group (it needs PARI's Galois data for degree $\le7$, built in for degree $4$). For each $p\nmid\operatorname{disc} f$, `factormod` gives the degrees. Cycle types of the groups, acting on $4$ points: $C_4$: $e$, $(2,2)$, two $4$-cycles; $V_4$: $e$, three $(2,2)$; $D_4$: $e$, two transpositions, three $(2,2)$, two $4$-cycles; $A_4$: $e$, three $(2,2)$, eight $3$-cycles; $S_4$: $1,6,3,8,6$.

In [ ]:
PATTERNS = [(1,1,1,1), (1,1,2), (2,2), (1,3), (4,)]
GROUPS = {
    "C4": {(1,1,1,1): 1, (2,2): 1, (4,): 2},
    "V4": {(1,1,1,1): 1, (2,2): 3},
    "D4": {(1,1,1,1): 1, (1,1,2): 2, (2,2): 3, (4,): 2},
    "A4": {(1,1,1,1): 1, (2,2): 3, (1,3): 8},
    "S4": {(1,1,1,1): 1, (1,1,2): 6, (2,2): 3, (1,3): 8, (4,): 6},
}
def probs(G):
    tot = sum(GROUPS[G].values())
    return {pt: GROUPS[G].get(pt, 0) / tot for pt in PATTERNS}

def pattern(f, p):
    fa = pari.factormod(f, p)
    degs = []
    for i in range(len(fa[0])):
        degs += [int(pari.poldegree(fa[0][i]))] * int(fa[1][i])
    return tuple(sorted(degs))

P = [int(q) for q in pari.primes(pari.primepi(10**5))]
for pol, G in [("x^4-2", "D4"), ("x^4+x+1", "S4"), ("x^4+8*x+12", "A4"), ("x^4-5*x^2+5", "C4")]:
    f = pari(pol); D = int(pari.poldisc(f))
    cnt = Counter(pattern(f, p) for p in P if D % p)
    N = sum(cnt.values())
    print(pol, "polgalois:", pari.polgalois(f)[3], "| expected", G)
    for pt in PATTERNS:
        print(f"   {str(pt):14s} observed {cnt[pt]/N:.4f}   Chebotarev {probs(G)[pt]:.4f}")

Every group matches its prediction to two or three decimals. Note how $C_4$ and $A_4$ never show a transposition, and $V_4$ (e.g. $X^4-10X^2+1$, the polynomial of $\sqrt2+\sqrt3$) is reducible modulo every prime: the phenomenon of Exercise 8.5(a) in disguise.

### P8.4
**How to find it.** (1) A factorisation over $\mathbb Q$ into degrees $(1,3)$ forces every pattern mod $p$ to contain a $1$; degrees $(2,2)$ force patterns refining $(2,2)$, so no $(1,3)$ and no $(4)$; degrees $(1,1,2)$ or finer force a $1$. So "a $(4)$" alone, or "$(1,3)$ and $(2,2)$" together, is incompatible with every proper factorisation. For $S_4$ the patterns $(4)$ have density $\frac14$, so the expected number of primes is about $4$; for $A_4$ there is no $(4)$, but $(2,2)$ and $(1,3)$ both occur. (2) Log-likelihood $\sum_i\log P_G(\text{pattern}_i)$, with $\log0=-\infty$; ties broken towards the smaller group.

In [ ]:
def irreducibility_certificate(f, maxp=10**5):
    seen = set(); D = int(pari.poldisc(f)); used = 0
    for p in P:
        if p > maxp: break
        if D % p == 0: continue
        used += 1
        pt = pattern(f, p); seen.add(pt)
        if (4,) in seen or ((1,3) in seen and (2,2) in seen):
            return used
    return None

def random_quartic():
    while True:
        f = pari(f"x^4+{random.randint(-20,20)}*x^3+{random.randint(-20,20)}*x^2+{random.randint(-20,20)}*x+{random.randint(-20,20)}")
        if pari.polisirreducible(f) and int(pari.poldisc(f)) != 0:
            return f

uses = Counter()
for _ in range(300):
    f = random_quartic()
    uses[irreducibility_certificate(f)] += 1
print("primes needed for a certificate (random quartics):", sorted((k, v) for k, v in uses.items() if k)[:10], "| none found:", uses[None])
print("mean:", sum(k * v for k, v in uses.items() if k) / sum(v for k, v in uses.items() if k))

In [ ]:
SIZE = {"C4": 4, "V4": 4, "D4": 8, "A4": 12, "S4": 24}
def guess_group(samples):
    best, bestll = None, -float("inf")
    for G in sorted(GROUPS, key=lambda g: SIZE[g]):
        pr = probs(G); ll = 0.0
        for s in samples:
            if pr[s] == 0: ll = -float("inf"); break
            ll += math.log(pr[s])
        if ll > bestll + 1e-12:
            best, bestll = G, ll
    return best

NAMES = {"C(4) = 4": "C4", "E(4) = 2[x]2": "V4", "D(4)": "D4", "A4": "A4", "S4": "S4"}
def pari_group(f):
    return NAMES.get(str(pari.polgalois(f)[3]), str(pari.polgalois(f)[3]))

# a test set containing every group
tests = [pari(s) for s in ["x^4-5*x^2+5", "x^4-10*x^2+1", "x^4-2", "x^4+8*x+12", "x^4+x+1",
                           "x^4-3", "x^4+5*x^2+5", "x^4-x^2+1"]]
tests += [random_quartic() for _ in range(40)]
truth = [pari_group(f) for f in tests]
print(Counter(truth))
for N in [5, 10, 20, 40, 80, 160]:
    ok = 0
    for f, G in zip(tests, truth):
        D = int(pari.poldisc(f))
        samp = [pattern(f, p) for p in P[:600] if D % p][:N]
        ok += guess_group(samp) == G
    print(f"N = {N:3d} sampled primes: {ok}/{len(tests)} groups identified")

A handful of primes certifies irreducibility (about two on average: for $S_4$ a $4$-cycle appears with probability $\frac14$ and a $3$-cycle with probability $\frac13$, and the pair $(1,3)$, $(2,2)$ also works). One random quartic in a few hundred resists forever: when the group is $V_4$, every pattern is $(1,1,1,1)$ or $(2,2)$, exactly as for a product of two quadratics, so Frobenius sampling can never prove irreducibility and one needs another tool (a factorisation algorithm, or Hensel lifting as in Chapter 6). A few dozen primes identify the group. The hardest pair is $D_4$ versus $C_4$ (or $V_4$ versus $C_4$): the only difference lies in patterns of moderate probability, and the likelihood ratio grows only linearly in $N$. There is a real limit: groups with the same number of elements of each cycle type (for instance two non-conjugate subgroups of $S_n$ with equal "cycle index", which first occurs in degree $8$) can never be told apart by Frobenius statistics; that is why `polgalois` uses resolvents and why modern algorithms (Stauduhar's method, Fieker–Klüners) combine sampling with exact $p$-adic computations.

**References.** H. W. Lenstra Jr., *Algorithms in algebraic number theory*, Bull. AMS 26 (1992); C. Fieker and J. Klüners, *Computation of Galois groups of rational polynomials*, LMS J. Comput. Math. 17 (2014); J. D. Dixon, *Computing subfields in algebraic number fields*, J. Austral. Math. Soc. 49 (1990) (random primes for Galois groups).

### P8.5
**How to find it.** Reuse the counts of Figure 8.1 (representations by $a^2+5b^2$ and $2a^2+2ab+3b^2$, divided by $2$).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
Xmax = 10**5
def counts(form, N):
    a, b, c = form; r = np.zeros(N + 1, dtype=np.int64); R = int(math.isqrt(4 * N)) + 3
    for u in range(-R, R + 1):
        v = np.arange(-R, R + 1)
        q = a * u * u + b * u * v + c * v * v
        q = q[(q > 0) & (q <= N)]
        np.add.at(r, q, 1)
    return np.cumsum(r // 1) // 2
kappa = math.pi / math.sqrt(20)
xs = np.arange(1, Xmax + 1)
c0 = counts((1, 0, 5), Xmax)[1:]; c1 = counts((2, 2, 3), Xmax)[1:]
plt.figure(figsize=(8, 3.5))
plt.plot(xs[::50], ((c0 - kappa * xs) / np.sqrt(xs))[::50], lw=0.7, label="principal class")
plt.plot(xs[::50], ((c1 - kappa * xs) / np.sqrt(xs))[::50], lw=0.7, label="other class")
plt.axhline(0, color="k", lw=0.5); plt.legend(); plt.xlabel("x"); plt.ylabel("(A(x) - kappa x)/sqrt(x)")
plt.show()
print("max |error|/x^(1/4):", max(abs(c0 - kappa * xs) / xs**0.25), max(abs(c1 - kappa * xs) / xs**0.25))

Divided by $\sqrt x$ the error shrinks visibly: $O(\sqrt x)$ is not the truth. This is a lattice-point problem in an ellipse, Gauss's circle problem in disguise, where the error is known to be $O(x^{131/416})$ (Huxley) and conjectured to be $O(x^{1/4+\varepsilon})$; the normalised error by $x^{1/4}$ stays bounded in the range tested. The Lipschitz principle is crude but universal; the sharp exponents come from exponential sums (Tome 13).